In [4]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ================= PERMUTATION IMPORTANCE =================


# ================= SETTINGS =================
IN_PATH = (
    r"C:\Users\salma\Documents\GitHub\Machine-Learning\data\processed\H2_noduplicates.csv"
)  # your input file
OUT_PATH = (
    r"C:\Users\salma\Documents\GitHub\Machine-Learning\data\processed\H2_clean.csv"
)  # new cleaned file
DROP_DUPLICATES = True  # harmless if your file has no duplicates left
TARGET = "IsCanceled"

# ================= 1. LOAD + FIX NULLS =================
df = pd.read_csv(IN_PATH)
df.columns = df.columns.str.strip()

text_cols = df.select_dtypes(exclude="number").columns
for c in text_cols:
    df[c] = df[c].str.strip()  # remove spaces
df[text_cols] = df[text_cols].replace(
    r"(?i)^null$", np.nan, regex=True
)  # "NULL"/"null" -> NaN
print("Real nulls per column:\n", df.isnull().sum()[df.isnull().sum() > 0])

# ================= 2. DROP LEAKAGE =================
df = df.drop(columns=[
    "ReservationStatus", "ReservationStatusDate", "AssignedRoomType"
])

# ================= 3. MISSING VALUES =================
df["Children"] = df["Children"].fillna(0)
df["Country"] = df["Country"].fillna("Unknown")
df["has_agent"] = df["Agent"].notna().astype(int)
print("has_agent:\n", df["has_agent"].value_counts())  # must show 0 and 1
df = df.drop(columns=["Agent", "Company"])

# ================= 4. INVALID ROWS =================
df["total_guests"] = df["Adults"] + df["Children"] + df["Babies"]
df["total_nights"] = (
    df["StaysInWeekendNights"] + df["StaysInWeekNights"]
)
df = df[(df["total_guests"] > 0) & (df["total_nights"] > 0)]  # keep other zeros
df = df[(df["Babies"] <= 5) & (df["ADR"] < 1000)]

# ================= 5. ONE DATE FEATURE =================
months = {m: i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June", "July",
     "August", "September", "October", "November", "December"], 1)
}
df["arrival_date"] = pd.to_datetime(dict(
    year=df["ArrivalDateYear"],
    month=df["ArrivalDateMonth"].map(months),
    day=df["ArrivalDateDayOfMonth"]))
df["arrival_days"] = (
    df["arrival_date"] - df["arrival_date"].min()
).dt.days
df = df.drop(columns=[
    "ArrivalDateYear", "ArrivalDateMonth",
    "ArrivalDateWeekNumber", "ArrivalDateDayOfMonth"
])

# ================= 6. OTHER FEATURES =================
df["has_children"] = ((df["Children"] + df["Babies"]) > 0).astype(int)
df["adr_per_person"] = df["ADR"] / df["total_guests"]
df["is_prev_canceller"] = (df["PreviousCancellations"] > 0).astype(int)

keep = df["Country"].value_counts().nlargest(15).index
df["Country"] = df["Country"].where(df["Country"].isin(keep), "Other")

if DROP_DUPLICATES:
    df = df.drop_duplicates()

# ================= 7. SAVE NEW CSV =================
assert df.isnull().sum().sum() == 0, "There are still nulls!"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH, df.shape)

# ================= 8. LOAD CLEAN CSV + SPLIT =================
df = pd.read_csv(OUT_PATH, parse_dates=["arrival_date"])
y = df[TARGET]
X = df.drop(columns=[TARGET, "arrival_date"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
rf = RandomForestClassifier(
    n_estimators=200, n_jobs=-1, class_weight="balanced", random_state=42
)
pipe = Pipeline([("prep", pre), ("rf", rf)])
pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, pipe.predict(X_test), digits=3))

# ================= 9. PERMUTATION IMPORTANCE =================
perm = permutation_importance(
    pipe, X_test, y_test, scoring="roc_auc",
    n_repeats=5, n_jobs=-1, random_state=42
)

imp = pd.Series(perm.importances_mean, index=X_test.columns)
err = pd.Series(perm.importances_std, index=X_test.columns)

# table (always shown)
table = pd.DataFrame(
    {"importance": imp, "std": err}
).sort_values("importance", ascending=False)
print(table.round(4))

# plot
imp_sorted = imp.sort_values()
fig, ax = plt.subplots(figsize=(9, 8))
imp_sorted.plot.barh(xerr=err[imp_sorted.index], ax=ax)
ax.set_title("Permutation importance (test set, ROC-AUC drop)")
ax.set_xlabel("Drop in ROC-AUC when the feature is shuffled")
fig.tight_layout()
plt.show()

Real nulls per column:
 Children        4
Country         6
Agent        5587
Company     50646
dtype: int64
has_agent:
 has_agent
1    47841
0     5587
Name: count, dtype: int64


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\salma\\Documents\\GitHub\\Machine-Learning\\data\\processed\\H2_clean.csv'